---
title: Pemantauan perairan terbuka dan vegetasi di Danau Rawa Pening
short_title: Danau Rawa Pening
subject: Studi Kasus
subtitle: Membandingkan perairan terbuka dan respons vegetasi dengan Sentinel-2.
description: Membandingkan citra warna alami, MNDWI, dan NDVI pada pengamatan Sentinel-2 Level-2A terpilih setiap bulan sepanjang 2024 di Danau Rawa Pening.
keywords:
  - sentinel-2
  - rawa-pening
  - perairan-terbuka
  - vegetasi-permukaan
  - mndwi
  - ndvi
  - pemantauan-danau
---

## A. Latar belakang

Peraturan Presiden Nomor 60 Tahun 2021 menetapkan Rawa Pening sebagai salah satu dari 15 Danau Prioritas Nasional [(Perpres No. 60 Tahun 2021)](#ref-perpres-2021). Penetapan danau prioritas mempertimbangkan tekanan terhadap daerah tangkapan, sempadan, dan badan air; berkurangnya luas atau volume tampungan; sedimentasi; serta penurunan kualitas air dan keanekaragaman hayati.

Penelitian di Rawa Pening mendokumentasikan eutrofikasi yang mendorong pertumbuhan eceng gondok [(Wuryanta dan Murtiono, 2018)](#ref-wuryanta-2018) serta sedimentasi yang mengurangi volume tampungan efektif [(Mawandha dkk., 2024)](#ref-mawandha-2024). Penelitian lain menggunakan Sentinel-1, Sentinel-2, foto udara, dan model klasifikasi untuk memetakan eceng gondok pada Juni, Juli, dan Agustus 2024 [(Muammal dan Marsisno, 2025)](#ref-muammal-2025).

Studi kasus ini menggunakan satu pengamatan Sentinel-2 terpilih setiap bulan sepanjang 2024 untuk membandingkan citra warna alami, MNDWI, dan NDVI. Hasilnya berupa peta serta estimasi luas perairan terbuka dan tutupan vegetasi yang terdeteksi di dalam batas Danau Rawa Pening pada setiap tanggal pengamatan. Perbandingan tersebut menjadi titik awal untuk mengenali perbedaan tutupan permukaan dan mengulang pengamatan pada periode berikutnya.

*Catatan:* NDVI menunjukkan respons spektral vegetasi, tetapi tidak mengidentifikasi spesies. Identifikasi eceng gondok dan penyebab perubahan tutupan memerlukan data referensi serta pemeriksaan lapangan.

## B. Menyiapkan data

Langkah pertama adalah mengimpor seluruh library Python yang dibutuhkan untuk pengolahan data spasial, visualisasi peta, dan manipulasi data.

In [ ]:
from datacube import Datacube
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from odc.geo.geom import Geometry, point
from odc.geo.xr import rasterize


dc = Datacube(app="monitor_danau")

Bujur lintang berikut merupakan titik yang berada di tengah Danau Rawa Pening, Kabupaten Semarang, Jawa Tengah. Untuk mengubah titik menjadi AoI, titik ini diperluas menggunakan buffer `0.04` derajat, lalu diambil batas area kotaknya (`boundingbox`). Kemudian area tersebut divisualisasi pada peta guna memvalidasi AoI sebelum proses pengunduhan data dimulai.

In [ ]:
latitude = -7.290336
longitude = 110.435128

bbox = point(longitude, latitude, crs="EPSG:4326").buffer(0.04).boundingbox

OpenStreetMap menyediakan batas Danau Rawa Pening dalam bentuk poligon. Batas ini dipakai pada perhitungan luas perairan terbuka dan tutupan vegetasi, sedangkan AoI kotak tetap digunakan saat memuat dan mengolah data.

In [ ]:
osm_response = requests.get(
    "https://nominatim.openstreetmap.org/lookup",
    params={
        "osm_ids": "W24222566",
        "format": "jsonv2",
        "polygon_geojson": 1,
    },
    headers={"User-Agent": "inadc-jupyter-monitor-danau"},
    timeout=30,
)
osm_response.raise_for_status()

lake_boundary = Geometry(
    osm_response.json()[0]["geojson"], crs="EPSG:4326"
).to_crs("EPSG:32749")

*Sumber batas:* © OpenStreetMap contributors, ODbL [(OpenStreetMap contributors, t.t.)](#ref-openstreetmap).

AoI dan batas danau divisualisasikan bersama pada peta sebelum data dimuat.

In [ ]:
aoi_map = bbox.explore()
lake_boundary.explore(map=aoi_map)

Variabel query berikut memuat parameter yang dibutuhkan oleh datacube untuk memuat data.

In [ ]:
query = {
    "product": "s2_l2a",
    "x": (bbox.left, bbox.right),
    "y": (bbox.bottom, bbox.top),
    "time": ("2024-01-01", "2024-12-31"),
    "output_crs": "EPSG:32749", # Rawa Pening berada di zona UTM 49S.
    "group_by": "solar_day",
}

## C. Memilih pengamatan

Seperti pengamatan satelit optis pada umumnya, kondisi area kajian yang bebas awan sangat dibutuhkan agar informasi permukaan terbaca dengan akurat. Sentinel-2 telah menyediakan kanal *Scene Classification Layer* (SCL) untuk mengidentifikasi kualitas setiap piksel secara otomatis. SCL dan band analisis akan menggunakan grid 10 meter yang sama agar mask piksel dapat diterapkan secara langsung.

Seluruh kanal dimuat satu kali sebagai data tertunda (*lazy loading*). Pada tahap pemilihan tanggal, hanya SCL yang dibaca; band analisis baru dibaca setelah pengamatan terpilih.

In [ ]:
data = dc.load(
    measurements=["red", "green", "blue", "nir", "swir16", "scl"],
    resolution=(-10, 10),
    dask_chunks={},
    **query,
)
data

Berikut adalah kode ringkas untuk menampilkan definisi kelas/nilai dari kanal SCL dengan mengakses atribut `flags_definition`:

In [ ]:
data.scl.attrs["flags_definition"]["qa"]["values"]

Berdasarkan daftar kelas di atas, SCL dibaca satu kali. Nilai yang mewakili gangguan, seperti awan, bayangan awan, area gelap, dan piksel rusak disingkirkan agar hanya menyisakan piksel permukaan yang bersih.

In [ ]:
excluded_scl = [0, 1, 2, 3, 8, 9, 10, 11]
usable_pixels = ~data.scl.isin(excluded_scl)

usable_percentage = (
    usable_pixels.mean(dim=("y", "x")) * 100
).compute()

Data 1D `usable_percentage` dikonversi ke `DataFrame` dan dilengkapi kolom `month` sebagai dasar pengelompokan serta pengurutan per bulan:

In [ ]:
quality_table = usable_percentage.to_dataframe(name="usable_percentage").reset_index()
quality_table["month"] = quality_table["time"].dt.strftime("%Y-%m")

Tabel diurutkan dari persentase piksel bersih tertinggi untuk mempertahankan satu tanggal terbaik per bulan dan membuang sisanya, lalu diurutkan kembali sesuai urutan waktu.

In [ ]:
selected_table = (
    quality_table
    .sort_values("usable_percentage", ascending=False)
    .drop_duplicates("month")
    .sort_values("time")
    .reset_index(drop=True)
)
selected_table[["time", "usable_percentage"]]

In [ ]:
selected_times = selected_table["time"].to_numpy()

observations = (
    data[["red", "green", "blue", "nir", "swir16"]]
    .sel(time=selected_times)
    .compute()
)
observations

Sebelum analisis dimulai, citra diperiksa secara visual melalui tampilan *true color* (warna alami/RGB). Langkah ini membantu memastikan setiap bulan memiliki citra yang cukup bersih untuk dianalisis.

In [ ]:
# Pengaturan tata letak grafik dikemas dalam fungsi siap pakai berikut.
# Cukup jalankan sel ini, lalu lanjutkan ke pemanggilan fungsinya di bawah.
def plot_monthly_maps(data, title, boundary, show_colorbar=True, **plot_kwargs):
    count = min(data.sizes["time"], 12)
    columns = min(count, 4)
    rows = (count + columns - 1) // columns
    boundary_x, boundary_y = boundary.geom.exterior.coords.xy

    fig, axes = plt.subplots(
        nrows=rows,
        ncols=columns,
        figsize=(15, 3.5 * rows),
        squeeze=False,
    )
    fig.subplots_adjust(wspace=0.25, hspace=0.35)
    all_axes = axes.flatten()

    for index, ax in enumerate(all_axes[:count]):
        image = data.isel(time=index).plot.imshow(
            ax=ax,
            add_colorbar=False,
            add_labels=False,
            **plot_kwargs,
        )
        ax.plot(boundary_x, boundary_y, color="black", linewidth=0.8)
        ax.set_title(str(data.time.isel(time=index).values)[:10])
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.tick_params(axis="both", labelsize=8)
        ax.locator_params(axis="x", nbins=4)

    for ax in all_axes[count:]:
        ax.set_visible(False)

    if show_colorbar:
        colorbar = fig.colorbar(
            image,
            ax=all_axes[:count],
            location="right",
            shrink=0.5,
            pad=0.03,
        )
        colorbar.ax.tick_params(labelsize=8)
        fig.suptitle(title)
    else:
        fig.suptitle(title)

    plt.show()

In [ ]:
rgb = observations[["red", "green", "blue"]].to_array(dim="band")

plot_monthly_maps(
    rgb,
    "True Color",
    lake_boundary,
    show_colorbar=False,
    vmin=0,
    vmax=3000,
)

Hasil pemeriksaan menunjukkan bahwa Januari, November, dan Desember tidak baik untuk digunakan pada analisis berikutnya. Kondisi citra pada bulan-bulan tersebut dapat membuat luas perairan terbuka terkalkulasikan jauh lebih kecil, karena tutupan awan dan bukan karena perubahan lahan. Beberapa piksel pada bulan Februari, Maret dan April terdapat piksel yang tidak bersih, tetapi pengamatan pada bulan tersebut tetap diikutsertakan.

In [ ]:
# Bulan berikut dikeluarkan setelah pemeriksaan true color.
excluded_months = [
    "2024-01",
    "2024-11",
    "2024-12",
]

selected_month = selected_table[
    ~selected_table["month"].isin(excluded_months)
].reset_index(drop=True)

Tanggal pengamatan yang lolos digunakan pada analisis berikutnya. Mask SCL juga dibuat hanya untuk pengamatan tersebut.

In [ ]:
selected_times = selected_month["time"].to_numpy()

selected_observations = observations.sel(time=selected_times)

mask = (
    ~data.scl.sel(time=selected_times).isin(excluded_scl)
).compute()

:::{admonition} Uji Pemahaman
:class: tip

Metadata `eo:cloud_cover` dapat digunakan untuk memilih pengamatan dengan tutupan awan rendah. Namun, metadata tersebut tidak cocok dijadikan dasar pemilihan pada studi kasus ini. **Mengapa demikian?**
:::

## D. Menghitung MNDWI dan NDVI

Analisis ini membandingkan perairan terbuka dan vegetasi di Rawa Pening pada pengamatan yang lolos pemeriksaan kualitas. Tujuannya adalah melihat perbedaan pola perairan terbuka dan vegetasi antarbulan. Perbandingan ini relevan karena pertumbuhan eceng gondok merupakan salah satu persoalan di Rawa Pening dan dapat memengaruhi area perairan terbuka.

Luasan perairan terbuka dihitung dengan *Modified Normalized Difference Water Index* (MNDWI). MNDWI membandingkan band hijau dan SWIR1, mengikuti metode [(Xu, 2006)](#ref-xu-2006):

$$
\text{MNDWI} = \frac{\text{Green} - \text{SWIR1}}{\text{Green} + \text{SWIR1}}
$$

Untuk melengkapi pembacaan citra, NDVI memperlihatkan pola vegetasi pada pengamatan yang sama. Perhitungannya menggunakan band inframerah dekat dan merah, seperti diperkenalkan oleh [(Rouse dkk., 1974)](#ref-rouse-1974):

$$
\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}
$$

Nilai band diubah menjadi `float32` sebelum dihitung. Mask dari pemeriksaan SCL diterapkan agar MNDWI dan NDVI hanya dihitung pada piksel bersih.

In [ ]:
bands = selected_observations[
    ["red", "green", "blue", "nir", "swir16"]
].astype("float32")
masked = bands.where(mask)

mndwi = (masked.green - masked.swir16) / (masked.green + masked.swir16)
ndvi = (masked.nir - masked.red) / (masked.nir + masked.red)

Nilai MNDWI dan NDVI dari pengamatan terpilih kini siap dibandingkan. Pada bagian berikutnya, kedua indeks ditampilkan bersama citra warna alami agar pola perairan terbuka dan vegetasi lebih mudah dibaca.

## E. Analisis hasil pengamatan

Hasil pengamatan dibaca melalui peta MNDWI, NDVI, serta estimasi luas perairan terbuka dan tutupan vegetasi. Peta indeks memperlihatkan pola perairan terbuka dan vegetasi pada pengamatan yang lolos pemeriksaan kualitas. Estimasi luas merangkum perubahan jumlah piksel yang diklasifikasikan sebagai perairan terbuka atau tutupan vegetasi di dalam batas Danau Rawa Pening.

### E.1 Perbandingan visual

Pengamatan yang lolos pemeriksaan kualitas ditampilkan dalam dua grafik. Grafik pertama menunjukkan MNDWI, sedangkan grafik kedua menunjukkan NDVI. Setiap grafik menampilkan hingga 12 pengamatan agar pola pada bulan yang berbeda dapat dibandingkan.

In [ ]:
plot_monthly_maps(
    mndwi,
    "MNDWI",
    lake_boundary,
    cmap="RdBu",
    vmin=-1,
    vmax=1,
)

Peta MNDWI menunjukkan perairan terbuka dengan warna biru, dengan cakupan paling luas dan bersih pada pengamatan Juli dan Agustus 2024. Sebaliknya, nilai MNDWI negatif (warna kemerahan) terlihat menjorok masuk dari tepi timur dan selatan ke dalam batas danau, terutama pada Juni, September, dan Oktober. Hal ini mencirikan berkurangnya area perairan terbuka akibat tutupan vegetasi atau permukaan dangkal.

In [ ]:
plot_monthly_maps(
    ndvi,
    "NDVI",
    lake_boundary,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
)

Peta NDVI memperlihatkan kontras yang jelas antara badan air (warna jingga hingga kuning pucat bernilai negatif atau mendekati nol) dan tutupan vegetasi (warna hijau bernilai positif tinggi). Bercak hijau tampak menjorok masuk dari bagian tepi ke dalam batas poligon danau; terutama pada pengamatan Mei, Juni, September, dan Oktober, yang posisinya berimpitan dengan area MNDWI negatif pada peta sebelumnya. Pola tersebut menunjukkan bahwa penyempitan perairan terbuka di bagian tepi danau sejalan dengan kemunculan tutupan vegetasi permukaan, termasuk kemungkinan vegetasi air terapung. Identifikasi jenisnya tetap memerlukan data referensi lapangan.

```{tip} Citra Tunggal vs. Komposit Median
Kedua rangkaian peta tersebut menunjukkan bahwa sebaran perairan terbuka dan vegetasi terapung di Rawa Pening dapat berubah dari bulan ke bulan. Untuk membandingkan kondisi pada waktu pengamatan tertentu, analisis ini menggunakan satu citra dengan kualitas terbaik dari setiap bulan, bukan komposit median bulanan. Komposit median menggabungkan piksel dari beberapa tanggal dalam satu bulan. Pada objek yang berubah cepat, seperti hamparan eceng gondok, penggabungan tersebut dapat menghaluskan atau mengaburkan batas tutupan. Akibatnya, hasil komposit tidak selalu merepresentasikan kondisi dan estimasi luas pada satu tanggal tertentu.
```

### E.2 Perbandingan luas perairan terbuka dan tutupan vegetasi

Setelah peta MNDWI dan NDVI dibuat, luas perairan terbuka dan tutupan vegetasi dapat dihitung pada setiap tanggal pengamatan di dalam batas Danau Rawa Pening. Hasilnya kemudian bisa digunakan untuk membandingkan perubahan kondisi permukaan danau antarbulan.

Poligon Danau Rawa Pening dari OpenStreetMap digunakan sebagai mask danau. Dengan demikian, perhitungan luas berdasarkan indeks hanya dilakukan di dalam poligon danau.

In [ ]:
lake_mask = rasterize(lake_boundary, mndwi.odc.geobox)

Nilai ambang MNDWI ditetapkan sebesar 0. Piksel dengan nilai MNDWI di atas ambang tersebut diklasifikasikan sebagai perairan terbuka.

In [ ]:
mndwi_threshold = 0
open_water = (mndwi > mndwi_threshold) & lake_mask

Selanjutnya, nilai ambang NDVI ditetapkan sebesar 0,75. Piksel dengan nilai NDVI di atas ambang tersebut diklasifikasikan sebagai tutupan vegetasi [(Pádua dkk., 2022)](#ref-padua-2022).

In [ ]:
ndvi_threshold = 0.75
vegetation = (ndvi > ndvi_threshold) & ~open_water & lake_mask

Setelah klasifikasi dilakukan, jumlah piksel perairan terbuka dan tutupan vegetasi dihitung pada setiap tanggal pengamatan. Karena citra dianalisis pada resolusi 10 meter, setiap piksel mewakili area 100 m². Jumlah piksel kemudian dikonversi menjadi kilometer persegi.

In [ ]:
open_water_area_km2 = (
    open_water.sum(dim=("y", "x")) * 100 / 1_000_000
)
vegetation_area_km2 = (
    vegetation.sum(dim=("y", "x")) * 100 / 1_000_000
)

Grafik berikut membandingkan luas perairan terbuka dan tutupan vegetasi pada setiap tanggal pengamatan. Dua garis pada grafik menunjukkan perubahan kedua kelas permukaan tersebut antarbulan.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(
    open_water_area_km2.time.values,
    open_water_area_km2.values,
    marker="o",
    color="tab:blue",
    label="Perairan terbuka",
)
ax.plot(
    vegetation_area_km2.time.values,
    vegetation_area_km2.values,
    marker="o",
    color="tab:green",
    label="Tutupan vegetasi",
)

ax.set_xticks(open_water_area_km2.time.values)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.set_xlabel("Tanggal pengamatan")
ax.set_ylabel("Luas (km²)")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Grafik di atas memperlihatkan bahwa sepanjang Februari hingga September 2024, perubahan luas perairan terbuka dan tutupan vegetasi cenderung bergerak berlawanan: ketika tutupan vegetasi (NDVI > 0,75) meningkat hingga puncaknya pada Juni (~3,3 km²), luas perairan terbuka sedikit menyusut, dan kembali mencapai titik tertinggi pada Juli hingga Agustus (~12,1 km²) saat tutupan vegetasi berkurang. 

Namun pada akhir Oktober (2024-10-29), kedua kurva justru sama-sama menurun; perairan terbuka menyusut cukup tajam menjadi sekitar 10,9 km², sementara tutupan vegetasi juga ikut turun ke kisaran 2,0 km². Artinya, ada bagian di dalam batas danau yang tidak lagi terbaca sebagai perairan terbuka maupun vegetasi rapat. Kondisi seperti ini dapat muncul ketika permukaan tepian danau berubah menjadi tutupan peralihan, misalnya lumpur dangkal yang tersingkap atau vegetasi air yang mulai mengering. Penyebab pastinya tetap perlu diperiksa lebih lanjut menggunakan data lapangan, catatan tinggi muka air, atau citra beresolusi lebih tinggi.

## F. Pengembangan analisis untuk pemantauan danau

Notebook ini menyajikan contoh sederhana penggunaan citra Sentinel-2 untuk membandingkan perairan terbuka dan vegetasi di Rawa Pening. Analisis ini dapat dikembangkan lebih lanjut untuk penggunaan operasional maupun untuk mendukung pengelolaan danau. Beberapa kemungkinan pengembangan yang dapat dieksplorasi antara lain:

- **Memperpanjang deret pengamatan hingga beberapa tahun**. Pemilihan satu pengamatan terbaik setiap bulan dapat digunakan untuk melihat pola musiman dan perubahan dari tahun ke tahun.

- **Menguji nilai ambang batas MNDWI dan NDVI dengan data referensi.** Foto udara, citra beresolusi lebih tinggi, atau pengamatan lapangan dapat digunakan untuk menentukan nilai ambang yang sesuai bagi perairan terbuka dan tutupan vegetasi di Rawa Pening.

- **Mengintegrasikan data radar (Sentinel-1)**. Karena pengamatan optis rentan tertutup awan, seperti yang terjadi pada Januari, November, dan Desember; citra radar dapat dimanfaatkan untuk menjaga kontinuitas pemantauan sepanjang tahun sekaligus membantu memetakan tutupan eceng gondok.

- **Menggabungkan hasil dengan data hidrologi dan lapangan.** Data tinggi muka air, curah hujan, kualitas air, atau catatan lapangan dapat membantu menjelaskan faktor pemicu di balik perubahan luas perairan terbuka.

- **Menyiapkan tabel pemantauan berkala.** Perubahan luas perairan terbuka dan tutupan vegetasi dapat dilacak antarperiode dan dibaca bersama kondisi lapangan untuk menentukan area prioritas yang perlu ditangani lebih lanjut.

## Referensi

(ref-perpres-2021)=
1. Presiden Republik Indonesia. (2021). *Peraturan Presiden Republik Indonesia Nomor 60 Tahun 2021 tentang Penyelamatan Danau Prioritas Nasional*.

(ref-wuryanta-2018)=
2. Wuryanta, A., & Murtiono, U. H. (2018). *Eutrophication and Solving Effort in Lake Rawapening, Semarang District Central Java (A Spasial Approach)*. *Jurnal Geografi*, 15(1).

(ref-mawandha-2024)=
3. Mawandha, H. G., Putra, M. E. B., Nursafa, I., Yuliardi, K. C., Kesuma, L. M., & Rulisyani, D. S. (2024). *Evaluation of sediment transport estimates using Sediment Routing Analysis (SRA) model: study case of Rawa Pening Lake*. *Modeling Earth Systems and Environment*, 10(6), 7097–7112.

(ref-muammal-2025)=
4. Muammal, A. S., & Marsisno, W. (2025). *Detection and Mapping of Invasive Alien Plant Water Hyacinth using Satellite Imagery and Machine Learning (Case Study: Rawa Pening Lake, Indonesia)*. *Proceedings of The International Conference on Data Science and Official Statistics*, 2025(1), 266–280.

(ref-openstreetmap)=
5. OpenStreetMap contributors. (n.d.). *Rawa Pening* (way 24222566). Data tersedia di bawah lisensi ODbL.

(ref-xu-2006)=
6. Xu, H. (2006). *Modification of normalised difference water index (NDWI) to enhance open water features in remotely sensed imagery*. *International Journal of Remote Sensing*, 27(14), 3025–3033.

(ref-rouse-1974)=
7. Rouse, J. W., Jr., Haas, R. H., Schell, J. A., & Deering, D. W. (1974). *Monitoring vegetation systems in the Great Plains with ERTS*. *Third ERTS Symposium*, NASA SP-351, 309–317.

(ref-padua-2022)=
8. Pádua, L., Duarte, L., Antão-Geraldes, A. M., Sousa, J. J., & Castro, J. P. (2022). *Spatio-Temporal Water Hyacinth Monitoring in the Lower Mondego (Portugal) Using Remote Sensing Data*. *Plants*, 11(24), 3465.